# Otimização Optuna

In [1]:
from datetime import datetime
import pandas as pd
from src.Classification.Optimizer import ClassificationOptunaOptimizer
from src.Data.Processor import DataStreamProcessor

# Definição dos datasets
# categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
# categorias = ['Adaptação']
# categorias = ['Recorrência']
# categorias = ['Generalização']
categorias = ['Consistência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES),
]

modelos_para_otimizar = ['LB', 'HAT', 'ARF', 'HT']

def build_stream(dataset_path, selected_features=None):
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=selected_features)

    return processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=[
            'Source IP', 'Source Port', 'Destination IP',
            'Destination Port', 'Protocol', 'Inbound'
        ],
        imputation_method='mediana'
    )


In [2]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')

    for feat_name, selected_feats in configuracoes_features:
        stream, targets, features = build_stream(dataset_path, selected_features=selected_feats)

        for model_name in modelos_para_otimizar:
            optimizer = ClassificationOptunaOptimizer(
                stream=stream,
                n_trials=50,
                target_names=targets,
                n_runs=5
            )

            best_params = optimizer.optimize(
                model_name=model_name,
                warmup_instances=2000,
                experiment_name=nome_experimento,
                num_features=len(features),
                exec_id=id_execucao,
                window_evaluation=100
            )
print("Conclusão dos experimentos de classificação.")

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LB] Iniciando otimização focada no F1-Score (Binário) com 50 trials...
Trial 1/50 | F1: 92.8571 | Prec: 95.5882 | Rec: 90.2778 | Params: {'ensemble_size': 80}
Trial 2/50 | F1: 92.9577 | Prec: 94.2857 | Rec: 91.6667 | Params: {'ensemble_size': 60}
Trial 3/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 40}
Trial 4/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 20}
Trial 5/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 20}
Trial 6/50 | F1: 92.9577 | Prec: 94.2857 | Rec: 91.6667 | Params: {'ensemble_size': 60}
Trial 7/50 | F1: 92.7536 | Prec: 96.9697 | Rec: 88.8889 | Params: {'ensemble_size': 30}
Trial 8/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 10}
Trial 9/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 50}
Trial 10/50 | F1: 94.2029 | Prec: 98.4848 | Rec: 90.2778 | Params: {'ensemble_size': 10}
Trial 11/50 | F1: 93.6170 | Prec: 95.6522 | Re

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LB] Iniciando otimização focada no F1-Score (Binário) com 50 trials...
Trial 1/50 | F1: 92.7536 | Prec: 96.9697 | Rec: 88.8889 | Params: {'ensemble_size': 30}
Trial 2/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 10}
Trial 3/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 40}
Trial 4/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 40}
Trial 5/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 40}
Trial 6/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 50}
Trial 7/50 | F1: 94.2857 | Prec: 97.0588 | Rec: 91.6667 | Params: {'ensemble_size': 80}
Trial 8/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 10}
Trial 9/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 50}
Trial 10/50 | F1: 94.9640 | Prec: 98.5075 | Rec: 91.6667 | Params: {'ensemble_size': 10}
Trial 11/50 | F1: 94.2857 | Prec: 97.0588 | Re

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k


...


[LB] Iniciando otimização focada no F1-Score (Binário) com 50 trials...
Trial 1/50 | F1: 96.9819 | Prec: 98.3673 | Rec: 95.6349 | Params: {'ensemble_size': 10}
Trial 2/50 | F1: 96.7742 | Prec: 98.3607 | Rec: 95.2381 | Params: {'ensemble_size': 90}
Trial 3/50 | F1: 97.0737 | Prec: 98.7680 | Rec: 95.4365 | Params: {'ensemble_size': 40}
Trial 4/50 | F1: 97.1831 | Prec: 98.5714 | Rec: 95.8333 | Params: {'ensemble_size': 70}
Trial 5/50 | F1: 97.0737 | Prec: 98.7680 | Rec: 95.4365 | Params: {'ensemble_size': 40}
Trial 6/50 | F1: 96.9758 | Prec: 98.5656 | Rec: 95.4365 | Params: {'ensemble_size': 100}
Trial 7/50 | F1: 96.8655 | Prec: 98.7629 | Rec: 95.0397 | Params: {'ensemble_size': 50}
Trial 8/50 | F1: 97.0854 | Prec: 98.3707 | Rec: 95.8333 | Params: {'ensemble_size': 80}
Trial 9/50 | F1: 96.9819 | Prec: 98.3673 | Rec: 95.6349 | Params: {'ensemble_size': 10}
Trial 10/50 | F1: 97.0854 | Prec: 98.3707 | Rec: 95.8333 | Params: {'ensemble_size': 80}
Trial 11/50 | F1: 96.8718 | Prec: 98.5626 | R

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LB] Iniciando otimização focada no F1-Score (Binário) com 50 trials...
Trial 1/50 | F1: 98.3363 | Prec: 98.2772 | Rec: 98.3955 | Params: {'ensemble_size': 70}
Trial 2/50 | F1: 98.4362 | Prec: 98.3968 | Rec: 98.4757 | Params: {'ensemble_size': 80}
Trial 3/50 | F1: 98.3745 | Prec: 98.4337 | Rec: 98.3153 | Params: {'ensemble_size': 50}
Trial 4/50 | F1: 98.4362 | Prec: 98.3968 | Rec: 98.4757 | Params: {'ensemble_size': 80}
Trial 5/50 | F1: 98.8552 | Prec: 98.9944 | Rec: 98.7164 | Params: {'ensemble_size': 10}
Trial 6/50 | F1: 98.4362 | Prec: 98.3968 | Rec: 98.4757 | Params: {'ensemble_size': 80}
Trial 7/50 | F1: 98.3745 | Prec: 98.4337 | Rec: 98.3153 | Params: {'ensemble_size': 50}
Trial 8/50 | F1: 98.8552 | Prec: 98.9944 | Rec: 98.7164 | Params: {'ensemble_size': 10}
Trial 9/50 | F1: 98.7565 | Prec: 98.7565 | Rec: 98.7565 | Params: {'ensemble_size': 30}
Trial 10/50 | F1: 98.5949 | Prec: 98.6742 | Rec: 98.5158 | Params: {'ensemble_size': 100}
Trial 11/50 | F1: 98.8552 | Prec: 98.9944 | R

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LB] Iniciando otimização focada no F1-Score (Binário) com 50 trials...
Trial 1/50 | F1: 98.4882 | Prec: 98.9870 | Rec: 97.9944 | Params: {'ensemble_size': 70}
Trial 2/50 | F1: 98.4690 | Prec: 98.9073 | Rec: 98.0345 | Params: {'ensemble_size': 80}
Trial 3/50 | F1: 98.7118 | Prec: 99.0707 | Rec: 98.3554 | Params: {'ensemble_size': 30}
Trial 4/50 | F1: 98.4466 | Prec: 99.0260 | Rec: 97.8740 | Params: {'ensemble_size': 50}
Trial 5/50 | F1: 98.4466 | Prec: 99.0260 | Rec: 97.8740 | Params: {'ensemble_size': 50}
Trial 6/50 | F1: 98.5273 | Prec: 99.1071 | Rec: 97.9543 | Params: {'ensemble_size': 20}
Trial 7/50 | F1: 98.4466 | Prec: 99.0260 | Rec: 97.8740 | Params: {'ensemble_size': 50}
Trial 8/50 | F1: 98.4255 | Prec: 99.0654 | Rec: 97.7938 | Params: {'ensemble_size': 90}
Trial 9/50 | F1: 98.4882 | Prec: 98.9870 | Rec: 97.9944 | Params: {'ensemble_size': 70}
Trial 10/50 | F1: 98.3256 | Prec: 98.9042 | Rec: 97.7537 | Params: {'ensemble_size': 40}
Trial 11/50 | F1: 98.5460 | Prec: 99.2273 | Re

# Execução Default

In [3]:
from datetime import datetime
import pandas as pd
from src.Classification.Pipeline import ClassificationExperimentRunner
from src.Classification.Models import get_classification_models
from src.Data.Processor import DataStreamProcessor

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES),
]

def build_stream(dataset_path, selected_features=None):
    df = pd.read_csv(dataset_path)
    processor = DataStreamProcessor(logging=False, selected_features=selected_features)

    return processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=[
            'Source IP', 'Source Port', 'Destination IP',
            'Destination Port', 'Protocol', 'Inbound'
        ],
        imputation_method='mediana'
    )


In [4]:
default_params = {
    'LB': {
        'base_learner': None,
        'ensemble_size': 100,
        'minibatch_size': None,
        'number_of_jobs': None
    },
    'HAT': {
        'grace_period': 200,
        'split_criterion': 'InfoGainSplitCriterion', 
        'confidence': 0.001,
        'tie_threshold': 0.05, 
        'leaf_prediction': 'NaiveBayesAdaptive',
        'nb_threshold': 0, 
        'numeric_attribute_observer': 'GaussianNumericAttributeClassObserver',
        'binary_split': False, 
        'max_byte_size': 33554432, 
        'memory_estimate_period': 1000000,
        'stop_mem_management': True, 
        'remove_poor_attrs': False, 
        'disable_prepruning': True
    },
    'ARF': {
        'base_learner': None,
        'ensemble_size': 100,
        'max_features': 0.6,
        'lambda_param': 6.0,
        'minibatch_size': None,
        'number_of_jobs': 1,
        'drift_detection_method': None,
        'warning_detection_method': None,
        'disable_weighted_vote': False,
        'disable_drift_detection': False,
        'disable_background_learner': False
    },
    'HT': {
        'grace_period': 200,
        'split_criterion': 'InfoGainSplitCriterion', 
        'confidence': 0.001,
        'tie_threshold': 0.05, 
        'leaf_prediction': 'NaiveBayesAdaptive',
        'nb_threshold': 0, 
        'numeric_attribute_observer': 'GaussianNumericAttributeClassObserver',
        'binary_split': False, 
        'max_byte_size': 33554432, 
        'memory_estimate_period': 1000000,
        'stop_mem_management': True, 
        'remove_poor_attrs': False, 
        'disable_prepruning': True
    }
}

model_config = {
    'LB': {
        'param_key': 'lb_params',
        'output_name': 'LeveragingBagging'
    },
    'HAT': {
        'param_key': 'hat_params',
        'output_name': 'HoeffdingAdaptiveTree'
    },
    'ARF': {
        'param_key': 'arf_params',
        'output_name': 'AdaptiveRandomForest'
    },
    'HT': {
        'param_key': 'ht_params',
        'output_name': 'HoeffdingTree'
    }
}

modelos_para_executar = ['LB', 'HAT', 'ARF', 'HT']

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')

    for feat_name, selected_feats in configuracoes_features:
        stream, targets, features = build_stream(dataset_path, selected_features=selected_feats)
        schema = stream.get_schema()

        for model_name in modelos_para_executar:
            params = default_params[model_name].copy()
            cfg = model_config[model_name]

            def make_model(run_seed=None, model_name=model_name, params=params, cfg=cfg, schema=schema):
                kwargs = {cfg['param_key']: params}
                models = get_classification_models(
                    schema,
                    selected_models=[model_name],
                    run_seed=run_seed,
                    **kwargs
                )
                return models[cfg['output_name']]

            algoritmos = {
                cfg['output_name']: make_model
            }

            runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)

            runner.run_classification_evaluation(
                stream=stream,
                algorithms=algoritmos,
                window_evaluation=100,
                warmup_instances=2000,
                title=nome_experimento,
                algorithm_params=params,
                is_optimized=False,
                num_features=len(features),
                exec_id=id_execucao
            )


/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LeveragingBagging] Executando 5 rodada(s) prequencial(is)...

                                                                          METRICS REPORT | DEFAULT_FULLFEATURES                                                                          
Algorithm                | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP       | FN       | Time (s)    | Task           | Strategy     | Win_Eval  
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LeveragingBagging        | 93.5822 ± 0.4225 | 96.2053 ± 1.0982 | 91.1111 ± 0.6804 |  0.9359 ± 0.0043 |    3 |    7 |   29.1116 | classification | classifier   | 100       
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Cumulative CSV:
 - o

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LeveragingBagging] Executando 5 rodada(s) prequencial(is)...

                                                                           METRICS REPORT | DEFAULT_33FEATURES                                                                           
Algorithm                | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP       | FN       | Time (s)    | Task           | Strategy     | Win_Eval  
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LeveragingBagging        | 94.2857 ± 0.0000 | 97.0588 ± 0.0000 | 91.6667 ± 0.0000 |  0.9429 ± 0.0000 |    2 |    6 |   16.8553 | classification | classifier   | 100       
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Cumulative CSV:
 - o

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k


...


[LeveragingBagging] Executando 5 rodada(s) prequencial(is)...

                                                                           METRICS REPORT | DEFAULT_33FEATURES                                                                           
Algorithm                | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP       | FN       | Time (s)    | Task           | Strategy     | Win_Eval  
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LeveragingBagging        | 81.2079 ± 0.0835 | 98.2232 ± 0.0969 | 69.2174 ± 0.1100 |  0.8186 ± 0.0008 |    8 |  177 |  107.5667 | classification | classifier   | 100       
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Cumulative CSV:
 - o

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LeveragingBagging] Executando 5 rodada(s) prequencial(is)...

                                                                          METRICS REPORT | DEFAULT_FULLFEATURES                                                                          
Algorithm                | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP       | FN       | Time (s)    | Task           | Strategy     | Win_Eval  
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LeveragingBagging        | 97.4475 ± 0.3380 | 99.0271 ± 0.0482 | 95.9195 ± 0.6296 |  0.9697 ± 0.0039 |   25 |  108 |  304.4596 | classification | classifier   | 100       
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Cumulative CSV:
 - o

/home/tomas/repos/anomaly-and-classification-data-stream/.venv/lib/python3.14/site-packages/pandas/core/generic.py:7387: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  result[k] = res_k



[LeveragingBagging] Executando 5 rodada(s) prequencial(is)...

                                                                           METRICS REPORT | DEFAULT_33FEATURES                                                                           
Algorithm                | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP       | FN       | Time (s)    | Task           | Strategy     | Win_Eval  
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
LeveragingBagging        | 91.2203 ± 1.5644 | 99.1788 ± 0.0391 | 84.4833 ± 2.7157 |  0.9008 ± 0.0169 |   19 |  409 |  167.3441 | classification | classifier   | 100       
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Cumulative CSV:
 - o